# 2-1. 단일표본 t-검정과 독립표본 t-검정

- 단일표본 t-검정으로 성인 여성 키의 평균이 기준값 163cm와 다른지 확인한다.
- 독립표본 t-검정으로 A반과 B반의 평균 점수 차이를 확인한다.

## 이 실습의 학습 기준

이 노트북은 한 가지 함수의 결과만 확인하지 않고 다음 흐름으로 학습한다.

1. 데이터 구조에 맞는 검정 방법과 가정을 확인한다.
2. 같은 목적의 방법이 여러 개면 모두 실행해 결과를 비교한다.
3. 일부러 적합하지 않은 방법도 비교할 때는, 왜 최종 결론에 사용하지 않는지 밝힌다.
4. 전제검정 결과가 본 검정 선택에 어떻게 연결되는지 확인한다.
5. p-value뿐 아니라 차이의 방향과 크기까지 해석한다.
6. p-value가 `0.0000`으로 보이는 것은 반올림 결과이며, 실제 확률이 0이라는 뜻은 아니다.
7. 유의수준 α=0.05에서 `p-value < 0.05`면 귀무가설을 기각하고, `p-value >= 0.05`면 귀무가설을 기각하지 못한다.

In [1]:
# 셀 목적: 평균·표준편차, 표 데이터 처리와 통계검정에 필요한 라이브러리를 불러온다.
# 해석 포인트: 이후 모든 검정 결과는 statistic·pvalue 속성으로 나누어 해석한다.
import numpy as np          # 평균, 표준편차 등 수치 계산
import pandas as pd         # CSV 파일을 표 형태로 읽기
from scipy import stats     # t-검정, 정규성 검정 등 통계 함수

In [2]:
# 셀 목적: 텍스트 파일의 키 자료를 읽어 숫자형 리스트로 변환한다.
# 해석 포인트: splitlines()로 줄 단위 값을 분리하고 float 변환 후 일부 값과 표본크기를 점검한다.
# 텍스트 파일을 읽기 모드("r")로 연다.
with open("datas2/성인여성_키_데이터.txt", "r") as file:  # 텍스트 파일을 읽기 모드로 열고 블록 종료 시 자동으로 닫음
    # 빈 줄이 있어도 안전하게 처리하기 위해 splitlines()를 사용한다.
    height_text = file.read().splitlines()  # 텍스트 파일을 줄 단위 문자열 목록으로 분리

# 문자열로 읽힌 키 데이터를 실수(float) 리스트로 변환한다.
heights = list(map(float, height_text))  # 문자열 키 값을 실수형 목록으로 변환

print("앞에서 5개:", heights[:5])  # 파일에서 읽은 성인 여성 키 데이터의 앞 5개 값을 출력
print("데이터 개수:", len(heights))  # 정규성 검정과 단일표본 t-검정에 사용할 키 표본의 크기 n을 출력

앞에서 5개: [150.27, 142.94, 160.99, 157.48, 153.46]
데이터 개수: 25


## 1. 단일표본 t-검정

성인 여성 키 표본의 평균이 기준값 163cm와 통계적으로 다른지 확인한다.

- 귀무가설(H₀): 성인 여성 키의 평균은 163cm이다.
- 대립가설(H₁): 성인 여성 키의 평균은 163cm와 다르다.

In [3]:
# 셀 목적: 성인 여성 키 표본의 평균과 표본 표준편차를 계산한다.
# 해석 포인트: ddof=1은 모집단 분산을 추정하는 표본 표준편차를 사용한다는 뜻이다.
# 표본 키 데이터의 평균과 표본 표준편차를 계산한다.
height_mean = np.mean(heights)  # 키 표본의 산술평균을 계산
height_std = np.std(heights, ddof=1)  # 키 표본의 표본 표준편차를 계산

print(f"평균 키: {height_mean:.2f}cm")  # 성인 여성 키 표본의 산술평균을 cm 단위로 출력
print(f"표본 표준편차: {height_std:.2f}cm")  # 성인 여성 키 표본의 표본표준편차(ddof=1)를 cm 단위로 출력

평균 키: 156.93cm
표본 표준편차: 10.18cm


### 1-1. KS-test와 Shapiro-Wilk 정규성 검정 비교

단일표본 t-검정 전에 키 데이터가 정규분포에서 크게 벗어나지 않는지 확인한다.

- KS-test: 실제 데이터의 누적분포와 기준 정규분포의 누적분포를 비교한다.
- Shapiro-Wilk: 정렬된 데이터의 형태가 정규분포에서 기대되는 형태와 비슷한지 확인한다.
- 귀무가설(H₀): 키 데이터는 정규분포를 따른다.
- 대립가설(H₁): 키 데이터는 정규분포를 따르지 않는다.

여기서는 두 방법을 모두 경험하고 결과를 비교한다. 다만 아래 KS-test는 표본에서 계산한 평균과 표준편차를 기준분포에 사용하므로 학습용 비교다. 작은 표본의 정규성 판단은 Shapiro-Wilk 결과를 중심으로 살펴본다.

In [4]:
# 셀 목적: Shapiro-Wilk 검정으로 키 표본의 정규성 위반 증거를 확인한다.
# 해석 포인트: p-value>=0.05는 정규성을 증명한 것이 아니라 정규성 위반 증거가 부족하다는 뜻이다.
# Shapiro-Wilk 정규성 검정을 수행한다.
height_shapiro = stats.shapiro(heights)  # 키 표본의 Shapiro-Wilk 검정 결과를 계산

print(f"Shapiro-Wilk 검정통계량: {height_shapiro.statistic:.4f}")  # 키 데이터의 Shapiro-Wilk W 검정통계량을 출력
print(f"p-value: {height_shapiro.pvalue:.4f}")  # 키 데이터의 Shapiro-Wilk 정규성 검정 p-value를 출력

Shapiro-Wilk 검정통계량: 0.9536
p-value: 0.3014


In [5]:
# 셀 목적: 학습용 KS-test로 경험누적분포와 표본 평균·표준편차를 넣은 정규 CDF를 비교한다.
# 해석 포인트: 모수를 같은 표본에서 추정했으므로 표준 KS의 엄밀한 p-value 해석에는 Lilliefors 보정 등이 필요하다.
# KS-test로 표본의 누적분포와 정규분포의 누적분포를 비교한다.
height_ks = stats.kstest(  # 키 표본의 KS-test 결과를 계산
    heights,  # 정규성 검정을 수행할 키 표본 전달
    stats.norm.cdf,  # 비교 기준으로 정규분포 누적분포함수 전달
    args=(height_mean, height_std)  # 기준 정규분포 CDF에 평균과 표준편차를 전달
)  # 함수 호출에 전달할 인자 목록을 닫음

print(f"KS 검정통계량: {height_ks.statistic:.4f}")  # 키 데이터와 비교 정규분포 사이의 최대 누적분포 차이인 KS 통계량을 출력
print(f"p-value: {height_ks.pvalue:.4f}")  # 키 데이터의 KS 정규성 검정 p-value를 출력

KS 검정통계량: 0.1122
p-value: 0.8772


In [6]:
# 셀 목적: KS-test와 Shapiro-Wilk의 검정통계량·p-value·판정을 한 표로 비교한다.
# 해석 포인트: 두 결론이 같더라도 작은 표본의 주 판단은 검정력이 좋은 Shapiro-Wilk를 중심으로 해석한다.
# 두 정규성 검정의 결과를 같은 표에서 비교한다.
height_normality_comparison = pd.DataFrame({  # 키 정규성 검정 두 방법의 결과를 DataFrame으로 구성
    "검정 방법": ["KS-test", "Shapiro-Wilk"],  # 적용한 통계검정 방법 이름을 기록
    "검정통계량": [height_ks.statistic, height_shapiro.statistic],  # 귀무가설 기준에서 벗어난 정도를 나타내는 통계량 기록
    "p-value": [height_ks.pvalue, height_shapiro.pvalue]  # 귀무가설 아래에서 현재 이상 결과가 나올 확률 기록
})  # 딕셔너리 구성과 함수 호출을 함께 마침

height_normality_comparison["결론"] = height_normality_comparison["p-value"].apply(  # 유의수준 0.05 기준의 판정 문구를 추가
    lambda p: "정규성 위반 증거 부족" if p >= 0.05 else "정규성 위반 가능성"  # 각 p-value를 받아 0.05 기준의 해석 문구로 변환
)  # 함수 호출에 전달할 인자 목록을 닫음

display(height_normality_comparison.round(4))  # DataFrame 결과를 표 형태로 노트북에 표시

same_height_conclusion = (height_ks.pvalue >= 0.05) == (height_shapiro.pvalue >= 0.05)  # 두 정규성 검정의 0.05 기준 판정이 같은지 비교
print("두 검정의 결론 일치 여부:", same_height_conclusion)  # KS와 Shapiro-Wilk가 유의수준 0.05에서 같은 정규성 판정을 내렸는지 출력
print("본 실습의 주 판단: Shapiro-Wilk 결과를 중심으로 해석한다.")  # 작은 표본인 본 실습에서는 Shapiro-Wilk 결과를 주 판단으로 사용한다는 기준을 출력

,검정 방법,검정통계량,p-value,결론
0,KS-test,0.1122,0.8772,정규성 위반 증거 부족
1,Shapiro-Wilk,0.9536,0.3014,정규성 위반 증거 부족


두 검정의 결론 일치 여부: True
본 실습의 주 판단: Shapiro-Wilk 결과를 중심으로 해석한다.


### 1-2. 단일표본 t-검정 수행

정규성 검정 결과를 확인한 뒤, 유의수준 0.05에서 성인 여성 키의 평균이 163cm와 다른지 양측 검정한다.

In [7]:
# 셀 목적: 단일표본 t-검정으로 표본평균이 기준 평균 163cm와 다른지 양측 검정한다.
# 해석 포인트: p-value<0.05이면 평균=163이라는 귀무가설을 기각하고, 그 외에는 기각하지 못한다.
reference_mean = 163  # 단일표본 t-검정의 비교 기준 평균을 163cm로 설정

# 표본평균을 기준값 163cm와 비교한다.
one_sample_result = stats.ttest_1samp(heights, popmean=reference_mean)  # 키 평균과 기준값을 비교한 단일표본 t-검정 결과를 계산

print(f"t-통계량: {one_sample_result.statistic:.4f}")  # 표본평균과 기준값 163cm의 차이를 표준오차로 나눈 단일표본 t-통계량을 출력
print(f"p-value: {one_sample_result.pvalue:.4f}")  # 평균 키가 163cm라는 귀무가설에 대한 양측검정 p-value를 출력

if one_sample_result.pvalue < 0.05:  # 단일표본 t-검정 p-value가 0.05보다 작은지 판정
    print("귀무가설을 기각한다: 평균 키는 163cm와 통계적으로 다르다.")  # p-value가 0.05 미만일 때 평균 키가 163cm와 다르다는 기각 결론을 출력
else:  # 앞의 조건이 거짓일 때 사용할 대안 처리 시작
    print("귀무가설을 기각하지 못한다: 평균 키가 163cm와 다르다고 할 충분한 증거가 없다.")  # p-value가 0.05 이상일 때 평균 차이를 입증하지 못했다는 비기각 결론을 출력

t-통계량: -2.9798
p-value: 0.0065
귀무가설을 기각한다: 평균 키는 163cm와 통계적으로 다르다.


## 2. 독립표본 t-검정

A반과 B반은 서로 다른 학생으로 구성된 독립된 두 그룹이다.

- 귀무가설(H₀): A반과 B반의 평균 점수는 같다.
- 대립가설(H₁): A반과 B반의 평균 점수는 다르다.

In [8]:
# 셀 목적: A반과 B반 점수가 긴 형식으로 저장된 CSV 파일을 DataFrame으로 읽는다.
# 해석 포인트: head()로 열 이름과 한 행이 학생 한 명을 나타내는지 먼저 확인한다.
scores_df = pd.read_csv("datas2/반별_점수_type1.csv", encoding="euc-kr")  # 긴 형식의 반별 점수 CSV를 DataFrame으로 읽음
scores_df.head()  # 점수 데이터의 앞부분을 표시해 구조 확인

,반,점수
0,A,73
1,A,69
2,A,71
3,A,71
4,A,73


In [9]:
# 셀 목적: 반 열을 조건으로 A반과 B반 점수를 서로 독립된 NumPy 배열로 분리한다.
# 해석 포인트: 집단별 표본크기와 앞부분 값을 출력해 필터링이 올바른지 점검한다.
group_a = scores_df.loc[scores_df["반"] == "A", "점수"].to_numpy()  # 긴 형식 데이터에서 A반 점수만 NumPy 배열로 추출
group_b = scores_df.loc[scores_df["반"] == "B", "점수"].to_numpy()  # 긴 형식 데이터에서 B반 점수만 NumPy 배열로 추출

print("A반 인원:", len(group_a), "점수 일부:", group_a[:5])  # A반의 표본크기와 데이터 확인용 앞 5개 점수를 출력
print("B반 인원:", len(group_b), "점수 일부:", group_b[:5])  # B반의 표본크기와 데이터 확인용 앞 5개 점수를 출력

A반 인원: 20 점수 일부: [73 69 71 71 73]
B반 인원: 10 점수 일부: [63 56 73 61 55]


In [10]:
# 셀 목적: A반과 B반의 평균·표본 표준편차·평균 차이를 기술통계로 비교한다.
# 해석 포인트: 검정 전 실제 차이의 방향과 집단별 산포를 먼저 확인한다.
mean_a = np.mean(group_a)  # A반 점수의 산술평균을 계산
mean_b = np.mean(group_b)  # B반 점수의 산술평균을 계산
std_a = np.std(group_a, ddof=1)  # A반 점수의 표본 표준편차를 계산
std_b = np.std(group_b, ddof=1)  # B반 점수의 표본 표준편차를 계산

print(f"A반 평균: {mean_a:.2f}, 표준편차: {std_a:.2f}")  # A반 점수의 산술평균과 표본표준편차를 출력
print(f"B반 평균: {mean_b:.2f}, 표준편차: {std_b:.2f}")  # B반 점수의 산술평균과 표본표준편차를 출력
print(f"평균 차이(A반 - B반): {mean_a - mean_b:.2f}")  # 두 집단의 차이 방향을 확인하도록 A반 평균에서 B반 평균을 뺀 값을 출력

A반 평균: 70.55, 표준편차: 5.68
B반 평균: 64.10, 표준편차: 8.28
평균 차이(A반 - B반): 6.45


### 2-1. A반과 B반의 정규성 검정 비교

각 반에 KS-test와 Shapiro-Wilk를 모두 적용한다. KS-test는 방법을 경험하기 위한 비교이고, 표본 수가 작은 A반과 B반의 주 판단은 Shapiro-Wilk를 사용한다.

- 귀무가설(H₀): 해당 반의 점수는 정규분포를 따른다.
- 대립가설(H₁): 해당 반의 점수는 정규분포를 따르지 않는다.

In [11]:
# 셀 목적: 두 반 각각에 KS-test와 Shapiro-Wilk 검정을 적용해 정규성 결과를 비교한다.
# 해석 포인트: 반별 평균·표본표준편차는 KS 기준 정규분포의 loc·scale 인자로 사용한다.
group_normality_rows = []  # 반별 정규성 결과를 누적할 빈 목록 생성

for group_name, group_values in [("A반", group_a), ("B반", group_b)]:  # A반과 B반을 차례로 꺼내 같은 정규성 계산 반복
    group_mean = np.mean(group_values)  # 현재 반복 중인 반의 평균을 계산
    group_std = np.std(group_values, ddof=1)  # 현재 반복 중인 반의 표본 표준편차를 계산
    ks_result = stats.kstest(  # 현재 변수 또는 집단의 KS-test 결과를 계산
        group_values,  # 현재 반의 점수 배열을 KS-test에 전달
        stats.norm.cdf,  # 비교 기준으로 정규분포 누적분포함수 전달
        args=(group_mean, group_std)  # 기준 정규분포 CDF에 평균과 표준편차를 전달
    )  # 함수 호출에 전달할 인자 목록을 닫음
    shapiro_result = stats.shapiro(group_values)  # 현재 변수 또는 집단의 Shapiro-Wilk 결과를 계산

    group_normality_rows.extend([  # 두 검정 결과 행을 기존 결과 목록에 이어 붙임
        {  # 현재 검정 방법의 결과 한 행을 딕셔너리로 구성하기 시작
            "그룹": group_name,  # 결과가 어느 반에 해당하는지 기록
            "검정 방법": "KS-test",  # 적용한 통계검정 방법 이름을 기록
            "검정통계량": ks_result.statistic,  # 귀무가설 기준에서 벗어난 정도를 나타내는 통계량 기록
            "p-value": ks_result.pvalue  # 귀무가설 아래에서 현재 이상 결과가 나올 확률 기록
        },  # 현재 검정 결과 행을 닫고 다음 결과 행으로 이어감
        {  # 현재 검정 방법의 결과 한 행을 딕셔너리로 구성하기 시작
            "그룹": group_name,  # 결과가 어느 반에 해당하는지 기록
            "검정 방법": "Shapiro-Wilk",  # 적용한 통계검정 방법 이름을 기록
            "검정통계량": shapiro_result.statistic,  # 귀무가설 기준에서 벗어난 정도를 나타내는 통계량 기록
            "p-value": shapiro_result.pvalue  # 귀무가설 아래에서 현재 이상 결과가 나올 확률 기록
        }  # 현재 결과 행의 딕셔너리 구성을 마침
    ])  # 목록을 전달한 함수 호출을 마침

group_normality_comparison = pd.DataFrame(group_normality_rows)  # 반별 정규성 검정 결과 목록을 DataFrame으로 변환
group_normality_comparison["결론"] = group_normality_comparison["p-value"].apply(  # 유의수준 0.05 기준의 판정 문구를 추가
    lambda p: "정규성 위반 증거 부족" if p >= 0.05 else "정규성 위반 가능성"  # 각 p-value를 받아 0.05 기준의 해석 문구로 변환
)  # 함수 호출에 전달할 인자 목록을 닫음

display(group_normality_comparison.round(4))  # DataFrame 결과를 표 형태로 노트북에 표시

,그룹,검정 방법,검정통계량,p-value,결론
0,A반,KS-test,0.1331,0.8255,정규성 위반 증거 부족
1,A반,Shapiro-Wilk,0.9697,0.7485,정규성 위반 증거 부족
2,B반,KS-test,0.1588,0.9295,정규성 위반 증거 부족
3,B반,Shapiro-Wilk,0.8888,0.1646,정규성 위반 증거 부족


### 2-2. Levene 등분산성 검정

독립표본 t-검정에는 두 가지 방식이 있다.

- Student 독립표본 t-검정: 두 그룹의 분산이 같다고 가정한다.
- Welch 독립표본 t-검정: 두 그룹의 분산이 같다고 가정하지 않는다.

Levene 검정으로 어떤 방식을 주 결론에 사용할지 정한다.

- 귀무가설(H₀): A반과 B반의 점수 분산은 같다.
- 대립가설(H₁): A반과 B반의 점수 분산은 다르다.

In [12]:
# 셀 목적: Levene 검정으로 두 독립 집단의 등분산성 가정을 확인한다.
# 해석 포인트: p-value>=0.05이면 등분산 Student 검정을, p-value<0.05이면 Welch 검정을 선택하도록 표시한다.
levene_result = stats.levene(group_a, group_b)  # 집단 간 등분산성을 확인하는 Levene 검정 결과를 계산

print(f"Levene 검정통계량: {levene_result.statistic:.4f}")  # A반과 B반의 분산 차이를 검정한 Levene 통계량을 출력
print(f"p-value: {levene_result.pvalue:.4f}")  # 두 집단의 분산이 같다는 Levene 귀무가설의 p-value를 출력

equal_variance = levene_result.pvalue >= 0.05  # Levene 결과로 등분산 가정 사용 여부를 결정
print("Student t-검정을 주 결과로 사용:", equal_variance)  # Levene p-value가 0.05 이상이어서 Student t-검정을 선택했는지 논릿값으로 출력

Levene 검정통계량: 2.0331
p-value: 0.1650
Student t-검정을 주 결과로 사용: True


### 2-3. Student t-검정과 Welch t-검정 비교

두 방식을 모두 실행해 분산 가정에 따라 t-통계량, 자유도, p-value가 어떻게 달라지는지 확인한다. 최종 결론에는 앞의 Levene 결과로 선택한 방식을 사용한다.

In [13]:
# 셀 목적: Student와 Welch 독립표본 t-검정을 모두 실행해 가정과 결과 차이를 비교한다.
# 해석 포인트: 최종 결과는 Levene 판정에 연결하되 평균 차이·자유도·p-value를 함께 확인한다.
student_result = stats.ttest_ind(group_a, group_b, equal_var=True)  # 등분산을 가정한 Student 독립표본 t-검정 결과를 계산
welch_result = stats.ttest_ind(group_a, group_b, equal_var=False)  # 등분산을 가정하지 않는 Welch 독립표본 t-검정 결과를 계산

independent_comparison = pd.DataFrame({  # Student와 Welch t-검정 결과를 비교표로 구성
    "검정 방법": ["Student t-test", "Welch t-test"],  # 적용한 통계검정 방법 이름을 기록
    "등분산 가정": [True, False],  # 각 검정이 두 집단의 동일 분산을 가정하는지 표시
    "t-통계량": [student_result.statistic, welch_result.statistic],  # 평균 차이를 표준오차로 나눈 t 통계량 기록
    "자유도": [student_result.df, welch_result.df],  # 참조 t·카이제곱 분포의 모양을 결정하는 자유도 기록
    "p-value": [student_result.pvalue, welch_result.pvalue]  # 귀무가설 아래에서 현재 이상 결과가 나올 확률 기록
})  # 딕셔너리 구성과 함수 호출을 함께 마침
independent_comparison["결론"] = independent_comparison["p-value"].apply(  # 유의수준 0.05 기준의 판정 문구를 추가
    lambda p: "평균 차이 유의" if p < 0.05 else "평균 차이 증거 부족"  # 각 p-value를 받아 0.05 기준의 해석 문구로 변환
)  # 함수 호출에 전달할 인자 목록을 닫음

display(independent_comparison.round(4))  # DataFrame 결과를 표 형태로 노트북에 표시

if equal_variance:  # 등분산 판정이 참이면 Student 결과를 선택
    selected_test_name = "Student 독립표본 t-검정"  # Levene 결과로 선택한 독립표본 t-검정 이름을 저장
    independent_result = student_result  # Levene 결과에 따라 선택한 독립표본 검정 결과를 저장
else:  # 앞의 조건이 거짓일 때 사용할 대안 처리 시작
    selected_test_name = "Welch 독립표본 t-검정"  # Levene 결과로 선택한 독립표본 t-검정 이름을 저장
    independent_result = welch_result  # Levene 결과에 따라 선택한 독립표본 검정 결과를 저장

print("Levene 검정에 따라 선택한 방법:", selected_test_name)  # Levene 결과에 따라 실제로 선택한 Student 또는 Welch t-검정 이름을 출력
print(f"선택한 검정의 p-value: {independent_result.pvalue:.4f}")  # 선택된 독립표본 t-검정의 평균 차이 검정 p-value를 출력

,검정 방법,등분산 가정,t-통계량,자유도,p-value,결론
0,Student t-test,True,2.5129,28.0000,0.0180,평균 차이 유의
1,Welch t-test,False,2.2166,13.3832,0.0445,평균 차이 유의


Levene 검정에 따라 선택한 방법: Student 독립표본 t-검정
선택한 검정의 p-value: 0.0180


### 2-4. 독립표본 t-검정 결과

- A반 평균은 70.55점, B반 평균은 64.10점으로 차이는 6.45점이었다.
- KS-test와 Shapiro-Wilk 모두 두 반에서 정규성 위반의 충분한 증거를 보이지 않았다.
- Levene 검정 p-value는 0.1650이므로 Student 독립표본 t-검정을 주 결과로 선택했다.
- Student 검정의 p-value는 0.0180, Welch 검정의 p-value는 0.0445였다.
- 두 방식 모두 유의수준 0.05에서 평균 차이가 유의하다는 같은 결론을 보였지만 통계량과 p-value는 달랐다.
- 주 결과에 따라 A반과 B반의 평균 점수는 통계적으로 다르며, A반 평균이 더 높다고 해석한다.

## 3. 같은 데이터의 저장 형태 비교

`type1`은 반 이름과 점수가 행으로 쌓인 긴 형식이고, `type2`는 A반과 B반이 별도 열인 넓은 형식이다. 저장 형태가 달라도 같은 표본인지 확인한다.

In [14]:
# 셀 목적: 같은 A·B반 점수가 열별로 저장된 넓은 형식 CSV를 읽는다.
# 해석 포인트: 서로 다른 데이터 저장 구조가 분석 배열로 변환된 뒤 동일한지 확인하기 위한 단계다.
scores_wide_df = pd.read_csv(  # 넓은 형식의 반별 점수 CSV를 DataFrame으로 읽음
    "datas2/반별_점수_type2.csv",  # 결과표 또는 방법 목록에 사용할 이름 지정
    encoding="euc-kr"  # 파일의 한글 문자 인코딩 방식을 지정
)  # 함수 호출에 전달할 인자 목록을 닫음
scores_wide_df.head()  # 넓은 형식 점수 데이터의 앞부분을 표시

,A반,B반
0,73,63.0
1,69,56.0
2,71,73.0
3,71,61.0
4,73,55.0


In [15]:
# 셀 목적: 각 반 열의 결측값을 제거하고 NumPy 배열로 변환한다.
# 해석 포인트: 긴 형식에서 추출한 배열과 완전히 같은지 array_equal()로 검증한다.
group_a_wide = scores_wide_df["A반"].dropna().to_numpy()  # 넓은 형식 A반 열에서 결측값을 제거하고 배열로 변환
group_b_wide = scores_wide_df["B반"].dropna().to_numpy()  # 넓은 형식 B반 열에서 결측값을 제거하고 배열로 변환

print("A반 데이터 동일 여부:", np.array_equal(group_a, group_a_wide))  # 긴 형식과 넓은 형식에서 추출한 A반 점수 배열이 같은지 출력
print("B반 데이터 동일 여부:", np.array_equal(group_b, group_b_wide))  # 긴 형식과 넓은 형식에서 추출한 B반 점수 배열이 같은지 출력

A반 데이터 동일 여부: True
B반 데이터 동일 여부: True
